# Example 6.3 in CT+iDCI paper

This example explores the process by which feasible sets are systematically enriched. 

There is no copula transformation performed because we use regular/joint DCI solutions within the iDCI framework when a specific push-forward constraint involves a multi-dimensional QoI map learned from the data. The standard DCI approach automatically incorporates a CT in such cases as described by Theorem 3.2 in the paper.

## Setup libraries and helper functions

This assumes the [LUQ: Learning Uncertain Quantities](https://github.com/CU-Denver-UQ/LUQ/) package has been installed, e.g., using `pip install git+https://github.com/CU-Denver-UQ/LUQ`

In [ ]:
# The libraries we will use
import numpy as np
import scipy.io as sio

# importing LUQ
from luq.luq import *

# distributions for data-generating samples and comparing approx vs true solutions
from scipy.stats import norm, beta, multivariate_normal

# Gaussian KDE 
from scipy.stats import gaussian_kde as GKDE
from scipy.stats import entropy

# plotting
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import ipywidgets as wd

# colorblind friendly color palette
c = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00']

# Set up fontsizes for plots
plt_params = {'legend.fontsize': 14,
          'figure.figsize': (6.4, 4.8),
         'axes.labelsize': 16,
         'axes.titlesize': 16,
         'xtick.labelsize': 14,
         'ytick.labelsize': 14}
plt.rcParams.update(plt_params)

np.random.seed(123456)

from tqdm import tqdm

import matplotlib.colors as mcolors
from sklearn.preprocessing import MinMaxScaler

%matplotlib widget
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

### Define iDCI function

In [ ]:
def iterative_DCI(Q_pred_vals, observed_densities, num_epochs=1, r_init=1, 
                  pred_tol=1e-12, ratio_tol=0.1, KL_tol=1e-7, KL_update_rel_tol = 1e-2,
                  verbose=False, QoI_spaces=[]):
    '''
    Assume Q_pred_vals is a (num_samples x QoI_num) array of QoI vals 
    associated with initial parameter samples and observed_densities 
    is a list of observed QoI densities for each QoI map (coming from a
    data generating distribution)
    '''
    num_samples, num_QoI = np.shape(Q_pred_vals)

    if len(QoI_spaces) == 0:
        QoI_spaces = list(np.arange(num_QoI))
    
    true_obs_probs = []
    for d in range(len(QoI_spaces)):
        true_obs_probs.append(observed_densities[d](Q_pred_vals[:,QoI_spaces[d]].T))
    
    with tqdm(total=100) as pbar:    
        
        if np.size(r_init)==1:  # No initial weighting (possibly from prior iterations)
            rs = np.ones((num_samples, len(QoI_spaces), num_epochs))
            r_current = np.ones(num_samples)
        else:
            r_current = np.copy(r_init)
            rs = np.ones((num_samples, len(QoI_spaces), num_epochs))
            
        kl_from_observed_marginal = np.zeros((len(QoI_spaces), num_epochs))
        
        for k in range(num_epochs):
            
            if verbose is True:
                print('Starting epoch #', k)
            
            for d in range(len(QoI_spaces)):
                r_current /= np.mean(r_current)
                if verbose is True: 
                    print('Iteration #', d)
                    print('Some current r-values:', r_current[0:3])
                pred_dens = GKDE(Q_pred_vals[:,QoI_spaces[d]].T,
                                 weights=r_current)
                rs_update = np.where(pred_dens(Q_pred_vals[:,QoI_spaces[d]].T)>pred_tol,
                                        np.divide(observed_densities[d](Q_pred_vals[:,QoI_spaces[d]].T),
                                        pred_dens(Q_pred_vals[:,QoI_spaces[d]].T)), 0)
                r_current *= rs_update
                rs[:,d,k] = r_current
                if np.abs(np.mean(r_current)-1.0)>ratio_tol:
                    print('Outside of ratio tolerance at epoch #', k+1, ' iteration #', d+1)
                    return rs[:,:d+1,:k+1], k+1, d+1, kl_from_observed_marginal[:,k]
                pbar.update(100/(len(QoI_spaces)*num_epochs))
            
            for d in range(len(QoI_spaces)):
                pred_dens = GKDE(Q_pred_vals[:, QoI_spaces[d]].T, 
                                 weights=r_current)
                current_probs = pred_dens(Q_pred_vals[:,QoI_spaces[d]].T)
                kl_from_observed_marginal[d, k] = entropy(true_obs_probs[d], current_probs) / num_samples
            
            if verbose is True:
                print('Epoch #', k)
                print('KL divergences from observed marginals', kl_from_observed_marginal[:,k])
            
            if all( kl < KL_tol for kl in kl_from_observed_marginal[:,k]):
                print('KL divergences from observed marginals all within tolerance at epoch #', k+1)
                return rs[:,:,:k+1], k+1, d+1, kl_from_observed_marginal
            
            if k>0:
                kl_relative_update = np.abs(kl_from_observed_marginal[:,k-1] - kl_from_observed_marginal[:,k]) /\
                                    kl_from_observed_marginal[:,k-1]
                if all(kl_rel < KL_update_rel_tol for kl_rel in kl_relative_update):
                    print('KL divergences from observed marginals are not sufficiently updated at epoch #', k+1)
                    return rs[:,:,:k+1], k+1, d+1, kl_from_observed_marginal
                    
    return rs, k+1, d+1, kl_from_observed_marginal

### Create 2d marginal plot function to compare different solutions on different feasible sets

Note that we are changing the default ax1 and ax2 titles here since no CT occurs in this example (we use regular/joint DCI solutions that automatically incorporate CT for each separately multi-dimensional QoI map)

In [ ]:
def plot_2d_params_compare(fignum, rs1, rs2, rs3=None, which_params = [0, 1], 
                           cmap='viridis_r',
                           ax1_title = 'iDCI Solution #1', 
                           ax2_title = 'iDCI Solution #2',
                           ax3_title = None,
                           exact_dg = False,
                           fix_cbar_ranges = True,
                           savefig=False,
                           filename='file.png'):

    # Construct updated densities for particular iterations of epoch
    update_dens_1 = GKDE(params_init[:, which_params].T,
                              weights = rs1)
    update_dens_2 = GKDE(params_init[:, which_params].T,
                              weights = rs2)

    if exact_dg is False:
        dg_dens = GKDE(params_dg[:, which_params].T)
    else:
        dg_dens = Beta_dg_pdf
        
    num_points = 20

    if which_params[0]==0:
        x = np.linspace(0.64-0.064,0.64+0.064,num_points)
    elif which_params[0]==1:
        x = np.linspace(0.8-0.08,0.8+0.08,num_points)
    else:
        x = np.linspace(1-0.1,1+0.1,num_points)

    if which_params[1]==0:
        y = np.linspace(0.64-0.064,0.64+0.064,num_points)
    elif which_params[1]==1:
        y = np.linspace(0.8-0.08,0.8+0.08,num_points)
    else:
        y = np.linspace(1-0.1,1+0.1,num_points)
    
    X, Y = np.meshgrid(x, y)
    plotting_params = np.vstack([X.ravel(), Y.ravel()]).T
    
    # Plot settings defined as dict
    base_cmap = plt.get_cmap(cmap)
    shared_levels=20
    color_matrix = base_cmap(np.linspace(0, 1, shared_levels-1))
    
    # Force the first 4 rows (bottom 4 levels) to be white
    color_matrix[0, :] = [1.0, 1.0, 1.0, 1.0] # [R, G, B, Alpha]
    custom_cmap = mcolors.ListedColormap(color_matrix)
    # Plot settings defined as dict
    plot_dict = {'cmap' : custom_cmap}
    
    if (fix_cbar_ranges is True) and (rs3 is None):
        true_vals = dg_dens(params_init[:, which_params].T)
        up_vals_1 = update_dens_1(params_init[:, which_params].T)
        up_vals_2 = update_dens_2(params_init[:, which_params].T)
        true_max = np.max([true_vals.max(), up_vals_1.max(), up_vals_2.max()])
        true_min = np.min([true_vals.min(), up_vals_1.min(), up_vals_2.min()])
        plot_dict['vmin'] = true_min
        plot_dict['vmax'] = true_max
        shared_levels = np.linspace(true_min, true_max, shared_levels)
    else:
        true_vals = dg_dens(params_init[:, which_params].T)
        up_vals_1 = update_dens_1(params_init[:, which_params].T)
        up_vals_2 = update_dens_2(params_init[:, which_params].T)
        
        update_dens_3 = GKDE(params_init[:, which_params].T,
                              weights = rs3)
        up_vals_3 = update_dens_3(params_init[:, which_params].T)
        
        true_max = np.max([true_vals.max(), up_vals_1.max(), up_vals_2.max(), up_vals_3.max()])
        true_min = np.min([true_vals.min(), up_vals_1.min(), up_vals_2.min(), up_vals_3.min()])
        plot_dict['vmin'] = true_min
        plot_dict['vmax'] = true_max
        shared_levels = np.linspace(true_min, true_max, shared_levels)

    # Create plots
    if rs3 is None:
        fig, axs = plt.subplots(1, 3, num=fignum, figsize=(9, 3), 
                            sharey='row', constrained_layout=True, clear=True)
    else:
        fig, axs = plt.subplots(1, 4, num=fignum, figsize=(12, 3), 
                            sharey='row', constrained_layout=True, clear=True)

    x_str = r'$\lambda_' + str(which_params[0]+1) + '$'
    y_str = r'$\lambda_' + str(which_params[1]+1) + '$'
    
    ax1 = axs[0]
    ax1.set_aspect('equal', adjustable='box')
    cf = ax1.contourf(X, Y, update_dens_1(plotting_params.T).reshape(len(y), len(x)),
                      levels=shared_levels, alpha=1,
                      **plot_dict)
    ax1.grid(True)
    title_str = ax1_title
    ax1.set_title(title_str)
    ax1.set_xlabel(x_str)
    ax1.set_ylabel(y_str)
    
    ax2 = axs[1]
    ax2.set_aspect('equal', adjustable='box')
    cf = ax2.contourf(X, Y, update_dens_2(plotting_params.T).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
    ax2.grid(True)
    title_str = ax2_title
    ax2.set_title(title_str)
    ax2.set_xlabel(x_str)
    ax2.set_ylabel(y_str)

    ax3 = axs[2]
    ax3.set_aspect('equal', adjustable='box')

    if rs3 is None:
        cf = ax3.contourf(X, Y, dg_dens(plotting_params.T).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
        ax3.grid(True)
        if exact_dg is False:
            title_str = 'Sample DG Distribution'
        else:
            title_str = 'Exact DG Distribution'
        ax3.set_title(title_str)
        ax3.set_xlabel(x_str)
        ax3.set_ylabel(y_str)
        
        fig.colorbar(cf, ax=[ax1, ax2, ax3], shrink=0.8)
    else:
        cf = ax3.contourf(X, Y, update_dens_3(plotting_params.T).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
        ax3.grid(True)
        if ax3_title is None:
            title_str = 'No Name'
        else:
            title_str = ax3_title
        ax3.set_title(title_str)
        ax3.set_xlabel(x_str)
        ax3.set_ylabel(y_str)

        ax4 = axs[3]
        ax4.set_aspect('equal', adjustable='box')
        cf = ax4.contourf(X, Y, dg_dens(plotting_params.T).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
        ax4.grid(True)
        ax4.set_title('DG Distr.')
        ax4.set_xlabel(x_str)
        ax4.set_ylabel(y_str)
        fig.colorbar(cf, ax=[ax1, ax2, ax3, ax4], shrink=0.8)
        
    if savefig is True:
        plt.savefig(filename, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

### Create function to approximate the KL divergence between two KDEs

In [ ]:
def kl_divergence_kde_grid(kde_p, kde_q, n_grid=100, dim=2, eps=1e-12):
    """
    Approximate KL(P||Q) over the unit box [0,1]^dim using a regular grid.

    Parameters
    ----------
    kde_p, kde_q : scipy.stats.gaussian_kde
        KDE objects.
    n_grid : int
        Number of grid points in each dimension.
    dim : int
        Dimension of the unit box.
    eps : float
        Small positive constant to avoid log(0).

    Returns
    -------
    float
        Approximate KL(P||Q).
    """

    # Construct regular grid
    axes = [np.linspace(0.0, 1.0, n_grid) for _ in range(dim)]
    mesh = np.meshgrid(*axes, indexing="ij")
    points = np.vstack([m.ravel() for m in mesh])  # shape (dim, n_grid**dim)

    # Evaluate KDEs
    p = kde_p(points)
    q = kde_q(points)

    # Renormalize on the unit box
    h = 1.0 / (n_grid - 1)
    cell_volume = h**dim

    p /= np.sum(p) * cell_volume
    q /= np.sum(q) * cell_volume

    # Numerical stability
    p = np.maximum(p, eps)
    q = np.maximum(q, eps)

    return np.sum(p * np.log(p / q)) * cell_volume

## Load the initial dataset containing both model data (used to generate predicted data) and parameter samples from initial distribution

The `init_data_set` we use comes from the prior LUQ paper and requires the `mat73` package to load.

In [ ]:
# !pip install mat73  # For reading in Matlab 7.3 files

In [ ]:
import mat73 as mat73

In [ ]:
init_data_set = mat73.loadmat('example_3_data/prior.mat')

In [ ]:
init_data_set.keys()

In [ ]:
data_init = init_data_set['data']

In [ ]:
data_init.shape

In [ ]:
a0_init = init_data_set['a1']  # initial samples of first parameter
a1_init = init_data_set['a2']  # initial samples of second parameter
a2_init = init_data_set['a3']

### Now load/analyze the observation and data-generating datasets

The `obs_data_set` we use was generated specifically for the CT+iDCI paper and uses the `loadmat` method within `scipy.io` to load.

In [ ]:
obs_data_set = sio.loadmat('example_3_data/obs_data.mat')

In [ ]:
obs_data_set.keys()

In [ ]:
obs_data_set['Expression1']['field1'][0][0].shape

In [ ]:
data_obs_1 = obs_data_set['Expression1']['field1'][0][0]
data_obs_2 = obs_data_set['Expression1']['field2'][0][0]
data_obs_3 = obs_data_set['Expression1']['field3'][0][0]

In [ ]:
data_obs = np.zeros((3000, 63, 3))
data_obs[:,:,0] = data_obs_1
data_obs[:,:,1] = data_obs_2
data_obs[:,:,2] = data_obs_3

In [ ]:
data_obs.shape

In [ ]:
# This loads in the data-generating parameters used to create the obs_data_set
dg_params = sio.loadmat('example_3_data/beta_copula_dg_samples.mat')

In [ ]:
dg_params.keys()

In [ ]:
a0_obs = dg_params['a1'].flatten()  # samples of first parameter responsible for observed data
a1_obs = dg_params['a2'].flatten()  # samples of second parameter responsible for observed data
a2_obs = dg_params['a3'].flatten()

## Set the precision of observable data and number of QoI to learn from each simulated experiment using LUQ

In [ ]:
predicted_precision = 2
num_predicted = 500

observed_precision = 2
num_obs = 50

num_qoi = [2, 3, 1]

learn_list = []
num_sets = 3

# Order of experiments
set_order = [0, 1, 2]

for j in range(num_sets):
    predicted_data = np.around(data_init[:num_predicted, :, set_order[j]], predicted_precision)
    learn_list.append(LUQ(predicted_data))
    learn_list[j].num_clusters = None
    
    learn_list[j].learn_qois_and_transform(num_qoi=num_qoi[j])
    
    observed_data = np.around(data_obs[:num_obs, :, set_order[j]], observed_precision)
    learn_list[j].set_observations(observed_data)
    
    learn_list[j].classify_and_transform_observations()

In [ ]:
# The QoI values are held within the pred_maps and obs_maps lists

pred_maps, obs_maps = [], []
for j in range(num_sets):
    obs_maps_temp = learn_list[j].transform_observations()
    pred_maps.append(learn_list[j].predict_maps)
    obs_maps.append(obs_maps_temp)

In [ ]:
# Associated parameter arrays used for plotting
params_init = np.vstack((a0_init[:num_predicted], 
                         a1_init[:num_predicted], 
                         a2_init[:num_predicted])).T

params_dg = np.vstack((a0_obs[:num_obs], 
                        a1_obs[:num_obs], 
                        a2_obs[:num_obs])).T

## Compute solutions

### Use first experiment data

In [ ]:
obs_densities = []
num_QoI = 2
QoI_spaces = [[0,1]]
obs_densities.append(GKDE(obs_maps[0][0][:,:num_QoI].T))

In [ ]:
# The first solution is a regular DCI solution so just 1 epoch is used
num_epochs = 1   

rs_1, last_epoch, last_iter, kl_epochs = iterative_DCI(pred_maps[0][0], obs_densities, QoI_spaces=QoI_spaces,
                                               num_epochs = num_epochs, r_init = 1, ratio_tol=0.2,
                                                        KL_tol=1e-7)

### Use second experiment data to enrich feasible set

In [ ]:
num_QoI = 3
QoI_spaces.append([2,3,4])
obs_densities.append(GKDE(obs_maps[1][0][:,:num_QoI].T))    

In [ ]:
# We need to combine the learned QoI values from the first experiment with the second experiment.
# We call this "flattening" the learned QoI.
pred_maps_flat = np.array([pred_maps[0][0][:,0],
                  pred_maps[0][0][:,1],
                  pred_maps[1][0][:,0],
                  pred_maps[1][0][:,1],
                  pred_maps[1][0][:,2]]).T

In [ ]:
num_epochs = 100

rs_2, last_epoch, last_iter, kl_epochs = iterative_DCI(pred_maps_flat, obs_densities, QoI_spaces = QoI_spaces,
                                               num_epochs = num_epochs, r_init = rs_1[:,-1,-1], ratio_tol=0.2,
                                                        KL_tol=1e-7)

### Use third experiment to enrich to the final feasible set

In [ ]:
num_QoI = 1
QoI_spaces.append([5])
obs_densities.append(GKDE(obs_maps[2][0][:,:num_QoI].T))

In [ ]:
pred_maps_flat = np.array([pred_maps[0][0][:,0],
                  pred_maps[0][0][:,1],
                  pred_maps[1][0][:,0],
                  pred_maps[1][0][:,1],
                  pred_maps[1][0][:,2],
                  pred_maps[2][0][:,0]]).T

In [ ]:
num_epochs = 100

rs_3, last_epoch, last_iter, kl_epochs = iterative_DCI(pred_maps_flat, obs_densities, QoI_spaces = QoI_spaces,
                                               num_epochs = num_epochs, r_init = rs_2[:,-1,-1], ratio_tol=0.2,
                                                        KL_tol=1e-7)

### Generate iDCI solution from original initial distribution to compare to above results

In [ ]:
rs_4, last_epoch, last_iter, kl_epochs = iterative_DCI(pred_maps_flat, obs_densities, QoI_spaces = QoI_spaces,
                                               num_epochs = num_epochs, r_init = 1, ratio_tol=0.2,
                                                        KL_tol=1e-7)

### Always check diagnostics at the end of each epoch

In [ ]:
# Check diagnostics at the end of each epoch
print(np.mean(rs_1))
print(np.mean(rs_2[:,-1,:],axis=0))
print(np.mean(rs_3[:,-1,:],axis=0))
print(np.mean(rs_4[:,-1,:],axis=0))

### Plot results

In [ ]:
fignum = 1

plot_2d_params_compare(fignum, 
                       rs_1[:,-1,-1],
                       rs_2[:,-1,-1], 
                       rs_3[:,-1,-1], 
                       ax1_title = r'$\mathbb{P}_1$',
                       ax2_title = r'$\mathbb{P}_2$',
                       ax3_title = r'$\mathbb{P}_3$',
                       which_params=[0, 1],
                       savefig=False,
                       filename='dependencies_trommel_params_12.png')

In [ ]:
fignum = 2

plot_2d_params_compare(fignum, 
                       rs_1[:,-1,-1],
                       rs_2[:,-1,-1], 
                       rs_3[:,-1,-1], 
                       ax1_title = r'$\mathbb{P}_1$',
                       ax2_title = r'$\mathbb{P}_2$',
                       ax3_title = r'$\mathbb{P}_3$',
                       which_params=[0, 2],
                       savefig=False,
                       filename='dependencies_trommel_params_13.png')

In [ ]:
fignum = 3

plot_2d_params_compare(fignum, 
                       rs_1[:,-1,-1],
                       rs_2[:,-1,-1], 
                       rs_3[:,-1,-1], 
                       ax1_title = r'$\mathbb{P}_1$',
                       ax2_title = r'$\mathbb{P}_2$',
                       ax3_title = r'$\mathbb{P}_3$',
                       which_params=[1, 2],
                       savefig=False,
                       filename='dependencies_trommel_params_23.png')

### Compute KL divergences

In [ ]:
# Transform all data to [0,1]^3 to use the divergence grid function
scaler = MinMaxScaler(feature_range=(0, 1))

# Fit to data and transform it
scaled_params = scaler.fit_transform(params_init)

init_dens = GKDE(scaled_params.T)
up_dens_1 = GKDE(scaled_params.T, weights=rs_1[:,-1,-1])
up_dens_2 = GKDE(scaled_params.T, weights=rs_2[:,-1,-1])
up_dens_3 = GKDE(scaled_params.T, weights=rs_3[:,-1,-1])
up_dens_4 = GKDE(scaled_params.T, weights=rs_4[:,-1,-1])

dg_dens = GKDE(scaler.transform(params_dg).T)

In [ ]:
print('KL divergences from X to estimate of DG distribution')
print('X=uniform initial:   ', kl_divergence_kde_grid(dg_dens, init_dens, dim=3))
print('X=DCI soln, P_1:     ', kl_divergence_kde_grid(dg_dens, up_dens_1, dim=3))
print('X=iDCI soln, P_2:    ', kl_divergence_kde_grid(dg_dens, up_dens_2, dim=3))
print('X=iDCI soln, P_3:    ', kl_divergence_kde_grid(dg_dens, up_dens_3, dim=3))
print('X=iDCI directly from unif. init.:   ', kl_divergence_kde_grid(dg_dens, up_dens_4, dim=3))
print('The KL divergence from iDCI solution computed from unif. init. to progressive iDCI soln:')
print(kl_divergence_kde_grid(up_dens_3, up_dens_4, dim=3))

In [ ]:
# DG tests

control_kls = []
for i in range(50):
    params_dg_control = np.vstack((a0_obs[num_obs*(i+1):num_obs*(i+2)], 
                            a1_obs[num_obs*(i+1):num_obs*(i+2)], 
                            a2_obs[num_obs*(i+1):num_obs*(i+2)])).T
    dg_dens_control = GKDE(scaler.transform(params_dg_control).T)
    control_kls.append(kl_divergence_kde_grid(dg_dens, dg_dens_control, dim=3))

In [ ]:
print('Average KL divergence of DG approximations:  ', np.mean(control_kls))
print('Min and Max KL divergences of DG approximations:   ', np.min(control_kls), np.max(control_kls))